# Handling mixed variables
Rebuilt for pandas 3. Data: `data/titanic.csv` (891 Titanic passengers: `Cabin`, `Ticket`, a made-up `number` column, `Survived`).

In [ ]:
# libraries for tables and charts
import pandas as pd
import plotly.express as px

In [ ]:
# load the data; pandas 3 reads text columns with the new `str` dtype
df = pd.read_csv("data/titanic.csv")
print(df.dtypes)
df.head()

## 1. A column with a number in some rows and a category in others

In [ ]:
# the values of `number`: digits, plus the letter A (alone)
print(df["number"].unique())
px.bar(df["number"].value_counts(), title="Passengers travelling with")

In [ ]:
# numerical part: anything that is not a number becomes NaN (errors="coerce")
# Int64 (capital I) is pandas' nullable integer: whole numbers plus <NA>
num = pd.to_numeric(df["number"], errors="coerce")
df["number_numerical"] = num.astype("Int64")

# categorical part: keep the original text only where the number failed
df["number_categorical"] = df["number"].where(num.isna())

df[["number", "number_numerical", "number_categorical"]].head(6)

## 2. A cell that holds a category and a number: Cabin

In [ ]:
# 147 different cabins: too many categories to use directly
print(df["Cabin"].nunique())

# number part: the first run of digits (raw string r"..." for the regular expression)
df["cabin_num"] = pd.to_numeric(df["Cabin"].str.extract(r"(\d+)")[0]).astype("Int64")
# category part: the first letter, the deck
df["cabin_cat"] = df["Cabin"].str[0]

df.loc[[0, 1, 3, 27, 75, 292], ["Cabin", "cabin_num", "cabin_cat"]]

In [ ]:
# only 8 decks are left
print(df["cabin_cat"].unique())
px.bar(df["cabin_cat"].value_counts(), title="Passengers per deck")

## 3. Ticket

In [ ]:
# split each ticket at the spaces into pieces
parts = df["Ticket"].str.split()

# number: the last piece, if it is a number
df["ticket_num"] = pd.to_numeric(parts.str[-1], errors="coerce").astype("Int64")

# category: the first piece, unless it is all digits
first = parts.str[0]
df["ticket_cat"] = first.where(~first.str.isdigit())

df.loc[[0, 1, 2, 3, 115, 179], ["Ticket", "ticket_num", "ticket_cat"]]

In [ ]:
# 681 tickets have become 43 ticket categories
print(df["Ticket"].nunique(), df["ticket_cat"].nunique())
print(df["ticket_cat"].unique())

In [ ]:
# Extra: the same prefix is written in many ways (A/5, A/5., A./5., A.5.)
# drop dots and slashes, upper-case: 43 categories become 29
clean = df["ticket_cat"].str.replace(r"[./]", "", regex=True).str.upper()
print(clean.nunique())

## 4. Filling the gaps the split left

In [ ]:
# A meant alone, so the number of companions is 0
# a missing category becomes its own label, "missing"
df["number_numerical"] = df["number_numerical"].fillna(0)
for col in ["number_categorical", "cabin_cat", "ticket_cat"]:
    df[col] = df[col].fillna("missing")
df.head()

In [ ]:
# Extra: does the deck tell us anything? survival rate per deck
df.groupby("cabin_cat")["Survived"].agg(["mean", "size"]).round(2)